# TB Drug Resistance Exploratory Data Analysis

The system rests on two datasets. The first is a synthetic patient case base that supports case-based reasoning. The second is the CRyPTIC cohort of real isolates that supports resistance classification. This notebook gives an overview of both.

## Synthetic case base

This first part covers four things. It measures the outcome signal in each feature, which sets the similarity weights. It checks the class imbalance across the six resistance profiles, which drives retrieval coverage. It fixes the trivial baselines a case-based design must beat, namely always predicting success for the outcome and the modal regimen per profile. It also reviews the seed graph coverage and its gaps.

In [1]:
import ast
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# The notebook runs from the eda/ folder. Source modules sit in the sibling
# SRC/ and Evaluation/ folders, and the CRyPTIC tables in Datasets/.
ROOT = Path.cwd().parent
DATA = ROOT / "Datasets"
sys.path[:0] = [str(ROOT / "SRC"), str(ROOT / "Evaluation")]

from cbr_cases import generate_cases
from validation import baseline_accuracy

N_CASES = 1000
SEED = 42
SEVERITY = ["Susceptible", "MonoResistant", "PolyResistant", "MDR", "PreXDR", "XDR"]

frame = pd.DataFrame(generate_cases(N_CASES, SEED))
success = frame["outcome"].eq("success")
len(frame)

1000

## 1. Distribution sanity check
Proportions across the whole case base, computed independently of `CaseGenerator.distribution_summary`, act as a cross-check that the generation matches the targets.

In [2]:
pd.concat({
    "profile": frame["profile"].value_counts(normalize=True).round(3),
    "region": frame["region"].value_counts(normalize=True).round(3),
})

profile  Susceptible        0.500
         MDR                0.180
         MonoResistant      0.120
         PreXDR             0.080
         XDR                0.060
         PolyResistant      0.060
region   SE_Asia            0.277
         African            0.256
         W_Pacific          0.165
         Americas           0.107
         European           0.100
         E_Mediterranean    0.095
Name: proportion, dtype: float64

In [3]:
frame["outcome"].value_counts(normalize=True).round(3)

outcome
success          0.738
ltfu             0.094
death            0.091
failed           0.046
not_evaluated    0.031
Name: proportion, dtype: float64

In [4]:
pd.Series({
    "hiv_positive": frame["hiv_status"].eq("positive").mean(),
    "diabetes": frame["diabetes"].mean(),
    "previous_treatment": frame["previous_treatment"].mean(),
    "mean_age": frame["age"].mean(),
}).round(3)

hiv_positive           0.109
diabetes               0.155
previous_treatment     0.149
mean_age              39.753
dtype: float64

## 2. Class balance across the six resistance profiles
Counts, share, and success rate for each profile, ordered by clinical severity. The case base is mostly Susceptible, and the resistant categories (PolyResistant, PreXDR, XDR) are much smaller. That shapes a clear design point. Neighbor-based retrieval has abundant data for Susceptible cases and thin data for the rarer resistant classes, so data quality per profile varies rather than holding steady.

In [5]:
pd.DataFrame({
    "n": frame["profile"].value_counts(),
    "share": frame["profile"].value_counts(normalize=True).round(3),
    "success_rate": success.groupby(frame["profile"]).mean().round(3),
}).reindex(SEVERITY)

,n,share,success_rate
profile,,,
Susceptible,500,0.50,0.822
MonoResistant,120,0.12,0.742
PolyResistant,60,0.06,0.650
MDR,180,0.18,0.717
PreXDR,80,0.08,0.512
XDR,60,0.06,0.483


## 3. Outcome signal by feature
Success rate varies across the values of each candidate similarity feature. A feature with a wide success-rate spread carries outcome signal and earns a higher weight in the similarity metric, while a feature with little spread earns less. This grounds the weights in the data rather than in assumptions. Region is included on purpose. It shapes prevalence and demographics in the generator but does not act on the outcome directly, so a nearly flat region row points to a low weight.

In [6]:
frame["age_band"] = pd.cut(frame["age"], bins=[17, 34, 49, 64, 80],
                           labels=["18-34", "35-49", "50-64", "65-80"])


def outcome_signal(column):
    return success.groupby(frame[column], observed=True).mean().round(3)


features = ["hiv_status", "previous_treatment", "diabetes", "sex", "age_band", "region"]
pd.concat({c: outcome_signal(c) for c in features})

hiv_status          negative           0.753
                    positive           0.615
previous_treatment  False              0.759
                    True               0.617
diabetes            False              0.751
                    True               0.665
sex                 F                  0.801
                    M                  0.710
age_band            18-34              0.739
                    35-49              0.771
                    50-64              0.687
                    65-80              0.675
region              African            0.738
                    Americas           0.757
                    E_Mediterranean    0.674
                    European           0.700
                    SE_Asia            0.762
                    W_Pacific          0.745
Name: outcome, dtype: float64

## 4. Trivial baselines to beat
Two simple predictors set the floor a case-based design must clear. The headline pair comes from `validation.baseline_accuracy`, the same function used in the validation report, so the baseline shown here and the one the CBR is tested against in `validation.py` are identical by design rather than merely similar. The per-profile tables below give the context behind each headline figure.

**Outcome, always predict success.** Most cases succeed, so predicting success for everyone already scores high. The success rate per profile equals that baseline's accuracy, and the overall value matches the `outcome` figure from the shared function.

**Regimen, most frequent regimen per profile.** Each profile leans on a dominant regimen, so this scores well. When one regimen holds the whole profile (share 1.0) the prediction is trivially correct. When the regimens split across two or three options, a smarter model has room to add value.

In [7]:
records = (frame.assign(actual_success=success, actual_regimen=frame["regimen"])
           [["actual_success", "profile", "actual_regimen"]]
           .to_dict("records"))
baseline_accuracy(records)

{'outcome': 0.738, 'regimen': 0.81}

In [8]:
outcome_baseline = pd.Series({
    "overall": success.mean(),
    **success.groupby(frame["profile"]).mean().to_dict(),
}).round(3)
outcome_baseline.reindex(["overall", *SEVERITY])

overall          0.738
Susceptible      0.822
MonoResistant    0.742
PolyResistant    0.650
MDR              0.717
PreXDR           0.512
XDR              0.483
dtype: float64

In [9]:
def modal_regimen(series):
    return series.value_counts().idxmax()


def majority_share(series):
    return series.value_counts(normalize=True).max()


pd.DataFrame({
    "modal_regimen": frame.groupby("profile")["regimen"].agg(modal_regimen),
    "majority_share": frame.groupby("profile")["regimen"].agg(majority_share).round(3),
    "n_regimens": frame.groupby("profile")["regimen"].nunique(),
}).reindex(SEVERITY)

,modal_regimen,majority_share,n_regimens
profile,,,
Susceptible,2HRZE_4HR,1.000,1
MonoResistant,6REZ_Lfx,1.000,1
PolyResistant,Individualized_12mo,0.517,2
MDR,BPaLM,0.450,3
PreXDR,Individualized_18mo,0.650,2
XDR,BPaL,0.433,3


## 5. Seed knowledge-graph composition
A static overview of the hand-curated strains in `tb_ontology.py`, read from the same source blob the test suite parses. This keeps the overview database-free and aligned with `test_core`. It shows what the curated graph holds (profiles, genes, and geography) and, just as important, what it leaves out. Those gaps are real. A query for an absent combination should return no rows rather than a fabricated strain. The integrity of the profile against mutation relationships is checked in the test suite, not re-proven here, since this section is for exploration rather than validation.

In [10]:
tree = ast.parse((ROOT / "SRC" / "tb_ontology.py").read_text())
blobs = {}
for node in ast.walk(tree):
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id in ("strains", "strain_data", "mutations"):
                try:
                    blobs[target.id] = ast.literal_eval(node.value)
                except (ValueError, SyntaxError):
                    pass

country = {s["id"]: s["country"] for s in blobs["strains"]}
gene = {m["id"]: m["gene"] for m in blobs["mutations"]}

kg = pd.DataFrame([{
    "strain": r["strain"],
    "country": country.get(r["strain"]),
    "profile": r["profile"],
    "genes": sorted({gene.get(m) for m in r["mutations"]} - {None}),
} for r in blobs["strain_data"]])

kg["profile"].value_counts().reindex(SEVERITY)

profile
Susceptible       6
MonoResistant    22
PolyResistant     9
MDR              10
PreXDR            7
XDR               2
Name: count, dtype: int64

In [11]:
pd.Series({
    "strains": len(kg),
    "countries": kg["country"].nunique(),
    "distinct_genes": len({g for genes in kg["genes"] for g in genes}),
})

strains           56
countries         42
distinct_genes     8
dtype: int64

In [12]:
india = kg[kg["country"] == "India"]
india[["strain", "profile", "genes"]]

,strain,profile,genes
3,TB004,PolyResistant,"[embB, inhA]"
4,TB005,MonoResistant,"[inhA, katG]"
33,TB034,PolyResistant,"[inhA, pncA]"
51,TB052,Susceptible,[]


The sections above run on the synthetic case base alone and need no datasets. The sections below read the CRyPTIC and WHO data files described in DEPLOYMENT.md, so they need those files present to run.

## 6. CRyPTIC real-world isolates

The remaining sections describe the CRyPTIC Release 3.4.0 cohort. These are real *M. tuberculosis* isolates with whole-genome variants graded against the WHO 2023 mutation catalog and measured drug-susceptibility phenotypes. The cohort carries genotype and phenotype only, with no treatment outcomes.

In [13]:
from feature_engineering import flat, resistant_profile, drug_map

# Reuse the CRyPTIC profile derivation from feature_engineering so this notebook
# and the model-ready table cannot drift. drug_map maps the release three-letter
# drug codes to system drug names, and resistant_profile turns a phenotype table
# into one resistance profile per isolate.
drugs = drug_map(DATA / "DRUG_CODES.csv")

## 7. Data sources and scale
The analysis draws on five tables. EFFECTS grades each isolate's mutations against the WHO 2023 catalog as R, S, U, or F. PREDICTIONS gives the catalog's per-drug genotypic call. DST_MEASUREMENTS and UKMYC_PHENOTYPES hold the measured susceptibility. DRUG_CODES maps the three-letter drug codes to drug names. The genome-wide MUTATIONS table is left out because EFFECTS already provides the same variants in graded form. The counts below cover the four parquet tables.

In [14]:
files = ["EFFECTS.parquet", "PREDICTIONS.parquet",
         "DST_MEASUREMENTS.parquet", "UKMYC_PHENOTYPES.parquet"]
pd.Series({name: pq.ParquetFile(DATA / name).metadata.num_rows for name in files})

EFFECTS.parquet             1154127
PREDICTIONS.parquet          810615
DST_MEASUREMENTS.parquet     660961
UKMYC_PHENOTYPES.parquet     288904
dtype: int64

## 8. Measured resistance-profile balance
Resistance profile derived from the DST measured phenotypes using the system's class definitions. As a resistance-prediction cohort, CRyPTIC holds thousands of isolates in every resistant tier, including pre-XDR and XDR. That distribution differs sharply from population prevalence, where those tiers would be sparse. PolyResistant is the smallest class and the narrowest by definition.

In [15]:
dst = resistant_profile(DATA / "DST_MEASUREMENTS.parquet", "PHENOTYPE", drugs)
dst.value_counts().reindex(SEVERITY)

Susceptible      39484
MonoResistant     7707
PolyResistant      874
MDR              10335
PreXDR            4725
XDR               2463
Name: count, dtype: int64

## 9. Cross-assay label concordance
UKMYC is an independent MIC-based assay. Across the isolates measured by both methods, the derived profiles are compared. Concordance is high, and the disagreements gather in the resistant tiers rather than in Susceptible, which fits the idea that borderline resistance is where assays diverge.

In [16]:
ukmyc = resistant_profile(DATA / "UKMYC_PHENOTYPES.parquet", "BINARY_PHENOTYPE", drugs)
both = pd.DataFrame({"dst": dst, "ukmyc": ukmyc}).dropna()
pd.Series({"measured_by_both": len(both),
           "agreement": round((both["dst"] == both["ukmyc"]).mean(), 3)})

measured_by_both    21568.000
agreement               0.956
dtype: float64

In [17]:
both[both["dst"] != both["ukmyc"]]["dst"].value_counts().reindex(SEVERITY, fill_value=0)

dst
Susceptible        0
MonoResistant    178
PolyResistant     42
MDR              347
PreXDR           192
XDR              198
Name: count, dtype: int64

## 10. Resistance signal versus phylogenetic background
Non-synonymous variants in resistance genes are not always tied to resistance. Several common changes (gyrA E21Q, gyrA S95T, katG R463L) act as phylogenetic markers and appear in most isolates. The true resistance determinants surface once the focus narrows to the catalog's resistance-graded calls. The ranking below, built from graded variants, brings the well-known resistance mutations to the top, such as katG S315T, rpoB S450L, and embB M306V.

In [18]:
def resistance_mutations(path):
    eff = flat(pd.read_parquet(path, columns=["UNIQUEID", "GENE", "MUTATION", "PREDICTION"]), "GENE")
    r = eff[eff["PREDICTION"].astype(str) == "R"]
    gene = r["GENE"].astype("string").fillna("NA")
    mut = gene.str.cat(r["MUTATION"].astype("string").fillna("NA"), sep="_")
    return r.assign(mut=mut).drop_duplicates(["UNIQUEID", "mut"])


resistance = resistance_mutations(DATA / "EFFECTS.parquet")
resistance["mut"].value_counts().head(15)

mut
katG_S315T     17071
rpoB_S450L     12359
rpsL_K43R       8843
embB_M306V      5155
fabG1_c-15t     4833
embB_M306I      3739
gyrA_D94G       2823
rrs_a1401g      2519
rrs_a514c       1652
gyrA_A90V       1643
rpoB_D435V      1564
rpsL_K88R       1451
embB_Q497R      1400
eis_c-12t        784
rrs_c517t        751
Name: count, dtype: Int64

## 11. Genotypic catalog prediction versus measured phenotype
The genotype-based profile from the catalog is compared with the measured phenotype profile, both overall and by category. Agreement is strongest for the Susceptible and high-resistance tiers and weakest for PolyResistant and MonoResistant, which rest on a small number of borderline calls.

In [19]:
catalog = resistant_profile(DATA / "PREDICTIONS.parquet", "PREDICTION", drugs)
compare = pd.DataFrame({"measured": dst, "catalog": catalog}).dropna()
compare["correct"] = compare["measured"] == compare["catalog"]
pd.Series({"n": len(compare), "exact_agreement": round(compare["correct"].mean(), 3)})

n                  53735.000
exact_agreement        0.813
dtype: float64

In [20]:
compare.groupby("measured")["correct"].mean().reindex(SEVERITY).round(3)

measured
Susceptible      0.891
MonoResistant    0.622
PolyResistant    0.229
MDR              0.743
PreXDR           0.748
XDR              0.785
Name: correct, dtype: float64

## 12. Catalog coverage gap between PREDICTIONS and EFFECTS
The rule engine reads graded mutations from EFFECTS, so an isolate with no EFFECTS rows reaches the engine with an empty genotype and is scored below-MDR. A few isolates carry a catalog call in PREDICTIONS yet have no graded rows in EFFECTS, and those isolates form the coverage gap. The counts below measure the gap across the full cohort and, within it, the isolates the catalog still calls resistant.

This gap is a property of the data, not a count of engine mistakes. The classification validation scores the full labeled cohort, where the same missing-input defect surfaces as the engine-only errors. The coverage gap counts every isolate that reaches the engine without graded rows, while the engine-only errors count only the subset where that missing input changes the predicted tier, so the gap is the larger figure and the two should be read separately rather than as the same number.

In [21]:
graded = set(flat(pd.read_parquet(DATA / "EFFECTS.parquet", columns=["UNIQUEID"]),
                  "UNIQUEID")["UNIQUEID"].unique())
predictions = flat(pd.read_parquet(DATA / "PREDICTIONS.parquet",
                                   columns=["UNIQUEID", "PREDICTION"]), "UNIQUEID")

gap = predictions[~predictions["UNIQUEID"].isin(graded)]
gap_ids = gap["UNIQUEID"].unique()
resistance_call = gap.loc[gap["PREDICTION"].astype(str) == "R", "UNIQUEID"].nunique()

pd.Series({
    "graded_in_effects": len(graded),
    "predicted_in_catalog": predictions["UNIQUEID"].nunique(),
    "coverage_gap": len(gap_ids),
    "gap_with_resistance_call": resistance_call,
})

graded_in_effects           53864
predicted_in_catalog        54041
coverage_gap                  200
gap_with_resistance_call      117
dtype: int64

The catalog tier of the gap isolates shows what the engine missed for want of input rows. The breakdown spans all gap isolates, and any MDR, PreXDR, or XDR call is a resistant isolate the rule engine scored below-MDR because EFFECTS held no graded mutation for it. The resistant tiers sum to fewer isolates than the resistance-call count above, since a handful of isolates carry an R call on a drug that does not drive the profile and so resolve to Susceptible here.

In [22]:
catalog_gap = resistant_profile(DATA / "PREDICTIONS.parquet", "PREDICTION", drugs).reindex(gap_ids)
catalog_gap.dropna().value_counts().reindex(SEVERITY, fill_value=0)

Susceptible      90
MonoResistant     9
PolyResistant     4
MDR              45
PreXDR           27
XDR              25
Name: count, dtype: int64

## 13. WHO catalog grading and gene tier

The graph loader in `who_catalog.py` reads this workbook and writes a resistance edge for every row, with confidence taken from the gene tier. The catalog instead grades each variant for its association with resistance, from group 1 associated with resistance to group 5 not associated. Reading the same sheet shows the grading composition and how little the tier tells us about it, so the loader can be limited to the associated groups and can read confidence from the grading.

In [23]:
who = pd.read_excel(DATA / "WHO-UCN-TB-2023.7-eng.xlsx", sheet_name=0, header=2)
gradings = [c for c in who.columns if "grading" in str(c).lower()]
grading = next((c for c in gradings if "final" in str(c).lower()), gradings[0])
who[grading].value_counts(dropna=False)

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


FINAL CONFIDENCE GRADING
3) Uncertain significance     33906
4) Not assoc w R - Interim    12379
2) Assoc w R - Interim         1130
5) Not assoc w R                484
1) Assoc w R                    253
Name: count, dtype: int64

In [24]:
pd.crosstab(who["tier"], who[grading])

FINAL CONFIDENCE GRADING,1) Assoc w R,2) Assoc w R - Interim,3) Uncertain significance,4) Not assoc w R - Interim,5) Not assoc w R
tier,,,,,
1,253,1130,14958,4998,250
2,0,0,18948,7381,234


## 14. Variant nomenclature on both sides

To merge the catalog mutations with the seed mutations they need one id format. This buckets the catalog variant strings and the CRyPTIC mutation strings by shape, so the canonical token can match what both already use.

In [25]:
import re

def variant_shape(v):
    v = str(v)
    if any(t in v for t in ("del", "ins", "dup", "fs")):
        return "indel"
    if "p." in v:
        return "protein_hgvs"
    if "c." in v or ">" in v:
        return "nucleotide"
    if re.search(r"[A-Za-z]\d+[A-Za-z]", v):
        return "protein_short"
    return "other"


who_variant = who["variant"].fillna(who["mutation"]).astype(str)
pd.DataFrame({"variant": who_variant, "shape": who_variant.map(variant_shape)}) \
  .groupby("shape")["variant"].agg(n="size", examples=lambda s: list(s.head(3)))

,n,examples
shape,,
indel,3459,"[bacA_c.-81_-76dupTCGGTG, bacA_deletion, bacA_..."
nucleotide,20861,"[bacA_c.102G>A, bacA_c.1044G>A, bacA_c.105C>G]"
other,67,"[bacA_LoF, eis_LoF, whiB6_LoF]"
protein_hgvs,23741,"[bacA_p.Ala152Val, bacA_p.Ala170Thr, bacA_p.Al..."
protein_short,24,"[Rv2477c_LoF, Rv1979c_LoF, Rv1979c_LoF]"


In [26]:
eff = flat(pd.read_parquet(DATA / "EFFECTS.parquet", columns=["MUTATION"]), "MUTATION")
mutation = eff["MUTATION"].astype(str)
pd.DataFrame({"mutation": mutation, "shape": mutation.map(variant_shape)}) \
  .groupby("shape")["mutation"].agg(n="size", examples=lambda s: list(s.head(3)))

,n,examples
shape,,
indel,15827,"[110_del_a, -38_del_cgttgacggcctcgacattacgttga..."
other,38354,"[c-61t, c-61t, g-10a]"
protein_short,1099946,"[G406A, F320F, A205A]"


## 15. Seed mutations in the catalog nomenclature

Each seed mutation matched to the WHO catalog by gene and position, so the 23 seed ids can be rewritten to the canonical HGVS form and merge with the catalog nodes.

In [27]:
from who_catalog import WHOCatalog

AA3 = {"A": "Ala", "R": "Arg", "N": "Asn", "D": "Asp", "C": "Cys", "Q": "Gln",
       "E": "Glu", "G": "Gly", "H": "His", "I": "Ile", "L": "Leu", "K": "Lys",
       "M": "Met", "F": "Phe", "P": "Pro", "S": "Ser", "T": "Thr", "W": "Trp",
       "Y": "Tyr", "V": "Val"}


def protein_hgvs(m):
    ref, alt, pos = m.get("ref"), m.get("alt"), m.get("position")
    if ref in AA3 and alt in AA3 and isinstance(pos, int) and pos > 0:
        return f"{m['gene']}_p.{AA3[ref]}{pos}{AA3[alt]}"
    return None


who_gene = who["gene"].map(WHOCatalog._normalize_gene)
who_variant = who["variant"].astype(str)
seed = []
for m in blobs["mutations"]:
    same_gene = who_variant[who_gene == m["gene"]]
    near = same_gene[same_gene.str.contains(str(m["position"]), na=False)]
    seed.append({"seed_id": m["id"], "candidate": protein_hgvs(m),
                 "who_variants": sorted(near.unique())[:4]})
pd.DataFrame(seed)

,seed_id,candidate,who_variants
0,rpoB_p.Ser450Leu,rpoB_p.Ser450Leu,"[rpoB_c.450C>T, rpoB_p.Ser450Ala, rpoB_p.Ser45..."
1,rpoB_p.Asp435Val,rpoB_p.Asp435Val,"[rpoB_p.Asp435Ala, rpoB_p.Asp435Asn, rpoB_p.As..."
2,rpoB_p.His445Tyr,rpoB_p.His445Tyr,"[rpoB_p.His445Arg, rpoB_p.His445Asn, rpoB_p.Hi..."
3,rpoB_p.His445Asp,rpoB_p.His445Asp,"[rpoB_p.His445Arg, rpoB_p.His445Asn, rpoB_p.Hi..."
4,rpoB_p.Leu430Pro,rpoB_p.Leu430Pro,"[rpoB_c.2430G>C, rpoB_p.Leu430Arg, rpoB_p.Leu4..."
5,katG_p.Ser315Thr,katG_p.Ser315Thr,"[katG_p.Ser315Arg, katG_p.Ser315Asn, katG_p.Se..."
6,katG_p.Ser315Asn,katG_p.Ser315Asn,"[katG_p.Ser315Arg, katG_p.Ser315Asn, katG_p.Se..."
7,katG_p.Ser315Ile,katG_p.Ser315Ile,"[katG_p.Ser315Arg, katG_p.Ser315Asn, katG_p.Se..."
8,inhA_c.-15C>T,None,"[inhA_c.-154G>A, inhA_c.-156C>T, inhA_c.-158C>..."
9,inhA_c.-8T>C,None,"[inhA_c.-800_-787delATTTCGGCCCGGCC, inhA_c.-80..."
